# Section 11 project — Fulfilment analytics with Matplotlib reports

Run the setup once, then complete the five stages in order. Every transformation remains visible in the notebook and every report is saved as a PNG file.


## Section setup — run once

Create one Spark session and load Matplotlib with a non-interactive backend suitable for saving files.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType
from decimal import Decimal
from IPython.display import Image, display
import json, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

spark = (SparkSession.builder.master('local[2]')
         .appName('PySpark-Fulfilment-Project')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark:', spark.version, '| Matplotlib:', matplotlib.__version__)


## Stage 1 — Define contracts and inspect reference data

Prepare the explicit schemas and small reference datasets used by all three runs.


### Step 1 — Set project paths

Keep all reads and writes beneath the project folder.


In [ ]:
PROJECT = 'project'
os.makedirs(f'{PROJECT}/reports', exist_ok=True)
os.makedirs(f'{PROJECT}/destination', exist_ok=True)
print('Project folder:', os.path.abspath(PROJECT))


### Step 2 — Declare explicit string schemas

Landing fields remain strings until quality rules convert them.


In [ ]:
shipment_schema = StructType([StructField(name, StringType(), True) for name in [
    'shipment_id', 'order_date', 'warehouse_id', 'carrier_id', 'promised_date',
    'delivered_date', 'shipping_cost', 'package_weight', 'status',
]])
warehouse_schema = StructType([StructField(name, StringType(), True) for name in ['warehouse_id', 'warehouse_name', 'warehouse_city']])
carrier_schema = StructType([StructField(name, StringType(), True) for name in ['carrier_id', 'carrier_name', 'service_type']])


### Step 3 — Read and verify references

Reference keys must be unique before they are joined to shipment rows.


In [ ]:
warehouses = spark.read.schema(warehouse_schema).option('header', True).csv(f'{PROJECT}/reference/warehouses.csv')
carriers = spark.read.schema(carrier_schema).json(f'{PROJECT}/reference/carriers.json')
assert warehouses.groupBy('warehouse_id').count().filter('count > 1').count() == 0
assert carriers.groupBy('carrier_id').count().filter('count > 1').count() == 0
warehouses.show(truncate=False)
carriers.show(truncate=False)


### Paired literal SQL solution

Register both references and confirm key uniqueness with SQL.


In [ ]:
warehouses.createOrReplaceTempView('s11_warehouses')
carriers.createOrReplaceTempView('s11_carriers')
sql_warehouse_duplicates = spark.sql('SELECT warehouse_id, COUNT(*) AS rows FROM s11_warehouses GROUP BY warehouse_id HAVING COUNT(*) > 1')
sql_carrier_duplicates = spark.sql('SELECT carrier_id, COUNT(*) AS rows FROM s11_carriers GROUP BY carrier_id HAVING COUNT(*) > 1')
sql_warehouse_duplicates.show()
sql_carrier_duplicates.show()
assert sql_warehouse_duplicates.count() == 0
assert sql_carrier_duplicates.count() == 0


## Stage 2 — run-001: process CSV, publish and report

Run the complete path from source data to a saved Matplotlib dashboard.


### Step 1 — Read and inspect the source batch

Add the run identifier and source filename before validation.


In [ ]:
run_id = 'run-001'
source_path = f'{PROJECT}/source/run-001/shipments.csv'
raw = spark.read.schema(shipment_schema).option('header', True).csv(source_path)
raw = raw.withColumn('run_id', F.lit(run_id)).withColumn('source_file', F.input_file_name()).cache()
raw.show(truncate=False)
print('Raw rows:', raw.count())


### Step 2 — Deduplicate, convert and classify

Retain malformed rows with a readable quality reason.


In [ ]:
unique = raw.dropDuplicates(['shipment_id'])
parsed = (unique
    .withColumn('shipment_id', F.trim('shipment_id'))
    .withColumn('warehouse_id', F.trim('warehouse_id'))
    .withColumn('carrier_id', F.trim('carrier_id'))
    .withColumn('status', F.upper(F.trim('status')))
    .withColumn('order_date', F.expr('try_cast(order_date as date)'))
    .withColumn('promised_date', F.expr('try_cast(promised_date as date)'))
    .withColumn('delivered_date', F.expr('try_cast(delivered_date as date)'))
    .withColumn('shipping_cost', F.expr('try_cast(shipping_cost as decimal(12,2))'))
    .withColumn('package_weight', F.expr('try_cast(package_weight as decimal(12,2))')))
quality_reason = F.concat_ws('; ',
    F.when(F.col('shipment_id').isNull() | (F.col('shipment_id') == ''), 'missing shipment id'),
    F.when(F.col('order_date').isNull(), 'invalid order date'),
    F.when(F.col('promised_date').isNull(), 'invalid promised date'),
    F.when(F.col('shipping_cost').isNull() | (F.col('shipping_cost') < 0), 'invalid shipping cost'),
    F.when(F.col('package_weight').isNull() | (F.col('package_weight') <= 0), 'invalid package weight'),
    F.when(F.col('warehouse_id').isNull() | (F.col('warehouse_id') == ''), 'missing warehouse'),
    F.when(F.col('carrier_id').isNull() | (F.col('carrier_id') == ''), 'missing carrier'),
    F.when(F.col('status').isNull() | ~F.col('status').isin('DELIVERED', 'IN_TRANSIT', 'CANCELLED'), 'invalid status'),
    F.when((F.col('status') == 'DELIVERED') & F.col('delivered_date').isNull(), 'missing delivered date'))
classified = parsed.withColumn('reject_reason', quality_reason)
quality_rejected = (classified.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('QUALITY')).cache())
valid = classified.filter(F.col('reject_reason') == '').drop('reject_reason')
delivered = valid.filter(F.col('status') == 'DELIVERED')
print('Unique:', unique.count(), '| quality rejected:', quality_rejected.count(), '| valid:', valid.count(), '| delivered:', delivered.count())


### Step 3 — Join references and calculate delivery measures

Unknown references are rejected; matched deliveries receive delay and on-time values.


In [ ]:
joined = (delivered.join(F.broadcast(warehouses), 'warehouse_id', 'left')
          .join(F.broadcast(carriers), 'carrier_id', 'left'))
reference_reason = F.concat_ws('; ',
    F.when(F.col('warehouse_name').isNull(), 'unknown warehouse'),
    F.when(F.col('carrier_name').isNull(), 'unknown carrier'))
reference_checked = joined.withColumn('reject_reason', reference_reason)
reference_rejected = (reference_checked.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('REFERENCE')).cache())
accepted = (reference_checked.filter(F.col('reject_reason') == '').drop('reject_reason')
    .withColumn('delay_days', F.datediff('delivered_date', 'promised_date'))
    .withColumn('on_time', F.col('delay_days') <= 0)
    .withColumn('delivery_month', F.date_format('delivered_date', 'yyyy-MM'))
    .select('shipment_id', 'run_id', 'order_date', 'warehouse_id', 'warehouse_name', 'warehouse_city',
            'carrier_id', 'carrier_name', 'service_type', 'promised_date', 'delivered_date',
            'shipping_cost', 'package_weight', 'status', 'source_file', 'delay_days', 'on_time', 'delivery_month')
    .cache())
accepted.orderBy("shipment_id").show(truncate=False)
print('Reference rejected:', reference_rejected.count(), '| accepted:', accepted.count())


### Step 4 — Write run outputs and audit metrics

Overwrite only this run so a replay does not duplicate earlier data.


In [ ]:
rejected = quality_rejected.unionByName(reference_rejected, allowMissingColumns=True)
curated_path = f'{PROJECT}/destination/curated_shipments/run_id={run_id}'
rejected_path = f'{PROJECT}/destination/rejected_records/run_id={run_id}'
audit_path = f'{PROJECT}/destination/audit/run_id={run_id}'
accepted.drop('run_id').write.mode('overwrite').partitionBy('delivery_month').parquet(curated_path)
rejected.drop('run_id').write.mode('overwrite').partitionBy('reject_stage').parquet(rejected_path)
accepted_count = accepted.count()
on_time_count = accepted.filter(F.col('on_time')).count()
shipping_cost_total = accepted.agg(F.sum('shipping_cost').alias('value')).first()['value'] or Decimal('0')
metrics = {
    'run_id': run_id, 'raw_rows': raw.count(), 'duplicate_rows': raw.count() - unique.count(),
    'quality_rejected_rows': quality_rejected.count(), 'non_delivered_rows': valid.count() - delivered.count(),
    'reference_rejected_rows': reference_rejected.count(), 'rejected_rows': rejected.count(),
    'accepted_shipments': accepted_count, 'on_time_shipments': on_time_count,
    'late_shipments': accepted_count - on_time_count, 'shipping_cost': f'{shipping_cost_total:.2f}',
}
spark.createDataFrame([metrics]).drop('run_id').write.mode('overwrite').parquet(audit_path)
print(metrics)


### Step 5 — Read the destination and save a Matplotlib dashboard

Spark produces compact summaries; Matplotlib plots only those summary rows.


In [ ]:
destination = spark.read.parquet(curated_path)
status_rows = destination.groupBy('on_time').count().orderBy(F.desc('on_time')).collect()
carrier_rows = (destination.groupBy('carrier_name').agg(F.avg('delay_days').alias('average_delay')).orderBy('carrier_name').collect())
city_rows = (destination.groupBy('warehouse_city').agg(F.sum('shipping_cost').alias('shipping_cost')).orderBy('warehouse_city').collect())
status_labels = ['On time' if row['on_time'] else 'Late' for row in status_rows]
status_values = [row['count'] for row in status_rows]
carrier_labels = [row['carrier_name'] for row in carrier_rows]
carrier_values = [float(row['average_delay']) for row in carrier_rows]
city_labels = [row['warehouse_city'] for row in city_rows]
city_values = [float(row['shipping_cost']) for row in city_rows]
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].axis('off')
on_time_pct = on_time_count / accepted_count * 100 if accepted_count else 0
axes[0, 0].text(0.05, 0.75, f'Accepted shipments\n{accepted_count}', fontsize=16, weight='bold')
axes[0, 0].text(0.05, 0.43, f'On-time rate\n{on_time_pct:.1f}%', fontsize=16, weight='bold')
axes[0, 0].text(0.05, 0.11, f'Shipping cost\n{float(shipping_cost_total):,.2f}', fontsize=16, weight='bold')
axes[0, 1].bar(status_labels, status_values, color=['#2a9d8f', '#e76f51'][:len(status_labels)])
axes[0, 1].set_title('On-time and late shipments')
axes[0, 1].set_ylabel('Shipments')
axes[1, 0].bar(carrier_labels, carrier_values, color='#457b9d')
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('Average delay by carrier')
axes[1, 0].set_ylabel('Days')
axes[1, 0].tick_params(axis='x', rotation=20)
axes[1, 1].bar(city_labels, city_values, color='#f4a261')
axes[1, 1].set_title('Shipping cost by warehouse city')
axes[1, 1].set_ylabel('Cost')
axes[1, 1].tick_params(axis='x', rotation=20)
figure.suptitle(f'Fulfilment dashboard — {run_id}', fontsize=18, weight='bold')
figure.tight_layout(rect=[0, 0, 1, 0.95])
report_path = f'{PROJECT}/reports/{run_id}-dashboard.png'
figure.savefig(report_path, dpi=150, bbox_inches='tight')
plt.close(figure)
assert os.path.isfile(report_path) and os.path.getsize(report_path) > 0
display(Image(filename=report_path))


### Paired literal SQL solution

Rebuild the accepted delivered rows with temporary views and compare the important totals.


In [ ]:
raw.createOrReplaceTempView('run_001_raw')
warehouses.createOrReplaceTempView('s11_sql_warehouses')
carriers.createOrReplaceTempView('s11_sql_carriers')
sql_result = spark.sql("""
WITH numbered AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY shipment_id ORDER BY shipment_id) AS duplicate_position FROM run_001_raw
), typed AS (
  SELECT TRIM(shipment_id) AS shipment_id, run_id,
         TRY_CAST(order_date AS DATE) AS order_date, TRIM(warehouse_id) AS warehouse_id,
         TRIM(carrier_id) AS carrier_id, TRY_CAST(promised_date AS DATE) AS promised_date,
         TRY_CAST(delivered_date AS DATE) AS delivered_date,
         TRY_CAST(shipping_cost AS DECIMAL(12,2)) AS shipping_cost,
         TRY_CAST(package_weight AS DECIMAL(12,2)) AS package_weight,
         UPPER(TRIM(status)) AS status, source_file
  FROM numbered WHERE duplicate_position = 1
), valid_delivered AS (
  SELECT * FROM typed WHERE shipment_id IS NOT NULL AND shipment_id <> ''
    AND order_date IS NOT NULL AND promised_date IS NOT NULL
    AND shipping_cost >= 0 AND package_weight > 0
    AND warehouse_id IS NOT NULL AND warehouse_id <> ''
    AND carrier_id IS NOT NULL AND carrier_id <> ''
    AND status = 'DELIVERED' AND delivered_date IS NOT NULL
)
SELECT v.shipment_id, v.run_id, v.order_date, v.warehouse_id, w.warehouse_name, w.warehouse_city,
       v.carrier_id, c.carrier_name, c.service_type, v.promised_date, v.delivered_date,
       v.shipping_cost, v.package_weight, v.status, v.source_file,
       DATEDIFF(v.delivered_date, v.promised_date) AS delay_days,
       v.delivered_date <= v.promised_date AS on_time,
       DATE_FORMAT(v.delivered_date, 'yyyy-MM') AS delivery_month
FROM valid_delivered v
INNER JOIN s11_sql_warehouses w ON v.warehouse_id = w.warehouse_id
INNER JOIN s11_sql_carriers c ON v.carrier_id = c.carrier_id
ORDER BY v.shipment_id
""")
sql_result.show(truncate=False)
assert sql_result.count() == accepted.count()
sql_cost = sql_result.agg(F.sum('shipping_cost')).first()[0] or Decimal('0')
assert f'{sql_cost:.2f}' == metrics['shipping_cost']
for frame in (raw, quality_rejected, reference_rejected, accepted):
    frame.unpersist()


## Stage 3 — run-002: process JSON, publish and report

Run the complete path from source data to a saved Matplotlib dashboard.


### Step 1 — Read and inspect the source batch

Add the run identifier and source filename before validation.


In [ ]:
run_id = 'run-002'
source_path = f'{PROJECT}/source/run-002/shipments.json'
raw = spark.read.schema(shipment_schema).json(source_path)
raw = raw.withColumn('run_id', F.lit(run_id)).withColumn('source_file', F.input_file_name()).cache()
raw.show(truncate=False)
print('Raw rows:', raw.count())


### Step 2 — Deduplicate, convert and classify

Retain malformed rows with a readable quality reason.


In [ ]:
unique = raw.dropDuplicates(['shipment_id'])
parsed = (unique
    .withColumn('shipment_id', F.trim('shipment_id'))
    .withColumn('warehouse_id', F.trim('warehouse_id'))
    .withColumn('carrier_id', F.trim('carrier_id'))
    .withColumn('status', F.upper(F.trim('status')))
    .withColumn('order_date', F.expr('try_cast(order_date as date)'))
    .withColumn('promised_date', F.expr('try_cast(promised_date as date)'))
    .withColumn('delivered_date', F.expr('try_cast(delivered_date as date)'))
    .withColumn('shipping_cost', F.expr('try_cast(shipping_cost as decimal(12,2))'))
    .withColumn('package_weight', F.expr('try_cast(package_weight as decimal(12,2))')))
quality_reason = F.concat_ws('; ',
    F.when(F.col('shipment_id').isNull() | (F.col('shipment_id') == ''), 'missing shipment id'),
    F.when(F.col('order_date').isNull(), 'invalid order date'),
    F.when(F.col('promised_date').isNull(), 'invalid promised date'),
    F.when(F.col('shipping_cost').isNull() | (F.col('shipping_cost') < 0), 'invalid shipping cost'),
    F.when(F.col('package_weight').isNull() | (F.col('package_weight') <= 0), 'invalid package weight'),
    F.when(F.col('warehouse_id').isNull() | (F.col('warehouse_id') == ''), 'missing warehouse'),
    F.when(F.col('carrier_id').isNull() | (F.col('carrier_id') == ''), 'missing carrier'),
    F.when(F.col('status').isNull() | ~F.col('status').isin('DELIVERED', 'IN_TRANSIT', 'CANCELLED'), 'invalid status'),
    F.when((F.col('status') == 'DELIVERED') & F.col('delivered_date').isNull(), 'missing delivered date'))
classified = parsed.withColumn('reject_reason', quality_reason)
quality_rejected = (classified.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('QUALITY')).cache())
valid = classified.filter(F.col('reject_reason') == '').drop('reject_reason')
delivered = valid.filter(F.col('status') == 'DELIVERED')
print('Unique:', unique.count(), '| quality rejected:', quality_rejected.count(), '| valid:', valid.count(), '| delivered:', delivered.count())


### Step 3 — Join references and calculate delivery measures

Unknown references are rejected; matched deliveries receive delay and on-time values.


In [ ]:
joined = (delivered.join(F.broadcast(warehouses), 'warehouse_id', 'left')
          .join(F.broadcast(carriers), 'carrier_id', 'left'))
reference_reason = F.concat_ws('; ',
    F.when(F.col('warehouse_name').isNull(), 'unknown warehouse'),
    F.when(F.col('carrier_name').isNull(), 'unknown carrier'))
reference_checked = joined.withColumn('reject_reason', reference_reason)
reference_rejected = (reference_checked.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('REFERENCE')).cache())
accepted = (reference_checked.filter(F.col('reject_reason') == '').drop('reject_reason')
    .withColumn('delay_days', F.datediff('delivered_date', 'promised_date'))
    .withColumn('on_time', F.col('delay_days') <= 0)
    .withColumn('delivery_month', F.date_format('delivered_date', 'yyyy-MM'))
    .select('shipment_id', 'run_id', 'order_date', 'warehouse_id', 'warehouse_name', 'warehouse_city',
            'carrier_id', 'carrier_name', 'service_type', 'promised_date', 'delivered_date',
            'shipping_cost', 'package_weight', 'status', 'source_file', 'delay_days', 'on_time', 'delivery_month')
    .cache())
accepted.orderBy("shipment_id").show(truncate=False)
print('Reference rejected:', reference_rejected.count(), '| accepted:', accepted.count())


### Step 4 — Write run outputs and audit metrics

Overwrite only this run so a replay does not duplicate earlier data.


In [ ]:
rejected = quality_rejected.unionByName(reference_rejected, allowMissingColumns=True)
curated_path = f'{PROJECT}/destination/curated_shipments/run_id={run_id}'
rejected_path = f'{PROJECT}/destination/rejected_records/run_id={run_id}'
audit_path = f'{PROJECT}/destination/audit/run_id={run_id}'
accepted.drop('run_id').write.mode('overwrite').partitionBy('delivery_month').parquet(curated_path)
rejected.drop('run_id').write.mode('overwrite').partitionBy('reject_stage').parquet(rejected_path)
accepted_count = accepted.count()
on_time_count = accepted.filter(F.col('on_time')).count()
shipping_cost_total = accepted.agg(F.sum('shipping_cost').alias('value')).first()['value'] or Decimal('0')
metrics = {
    'run_id': run_id, 'raw_rows': raw.count(), 'duplicate_rows': raw.count() - unique.count(),
    'quality_rejected_rows': quality_rejected.count(), 'non_delivered_rows': valid.count() - delivered.count(),
    'reference_rejected_rows': reference_rejected.count(), 'rejected_rows': rejected.count(),
    'accepted_shipments': accepted_count, 'on_time_shipments': on_time_count,
    'late_shipments': accepted_count - on_time_count, 'shipping_cost': f'{shipping_cost_total:.2f}',
}
spark.createDataFrame([metrics]).drop('run_id').write.mode('overwrite').parquet(audit_path)
print(metrics)


### Step 5 — Read the destination and save a Matplotlib dashboard

Spark produces compact summaries; Matplotlib plots only those summary rows.


In [ ]:
destination = spark.read.parquet(curated_path)
status_rows = destination.groupBy('on_time').count().orderBy(F.desc('on_time')).collect()
carrier_rows = (destination.groupBy('carrier_name').agg(F.avg('delay_days').alias('average_delay')).orderBy('carrier_name').collect())
city_rows = (destination.groupBy('warehouse_city').agg(F.sum('shipping_cost').alias('shipping_cost')).orderBy('warehouse_city').collect())
status_labels = ['On time' if row['on_time'] else 'Late' for row in status_rows]
status_values = [row['count'] for row in status_rows]
carrier_labels = [row['carrier_name'] for row in carrier_rows]
carrier_values = [float(row['average_delay']) for row in carrier_rows]
city_labels = [row['warehouse_city'] for row in city_rows]
city_values = [float(row['shipping_cost']) for row in city_rows]
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].axis('off')
on_time_pct = on_time_count / accepted_count * 100 if accepted_count else 0
axes[0, 0].text(0.05, 0.75, f'Accepted shipments\n{accepted_count}', fontsize=16, weight='bold')
axes[0, 0].text(0.05, 0.43, f'On-time rate\n{on_time_pct:.1f}%', fontsize=16, weight='bold')
axes[0, 0].text(0.05, 0.11, f'Shipping cost\n{float(shipping_cost_total):,.2f}', fontsize=16, weight='bold')
axes[0, 1].bar(status_labels, status_values, color=['#2a9d8f', '#e76f51'][:len(status_labels)])
axes[0, 1].set_title('On-time and late shipments')
axes[0, 1].set_ylabel('Shipments')
axes[1, 0].bar(carrier_labels, carrier_values, color='#457b9d')
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('Average delay by carrier')
axes[1, 0].set_ylabel('Days')
axes[1, 0].tick_params(axis='x', rotation=20)
axes[1, 1].bar(city_labels, city_values, color='#f4a261')
axes[1, 1].set_title('Shipping cost by warehouse city')
axes[1, 1].set_ylabel('Cost')
axes[1, 1].tick_params(axis='x', rotation=20)
figure.suptitle(f'Fulfilment dashboard — {run_id}', fontsize=18, weight='bold')
figure.tight_layout(rect=[0, 0, 1, 0.95])
report_path = f'{PROJECT}/reports/{run_id}-dashboard.png'
figure.savefig(report_path, dpi=150, bbox_inches='tight')
plt.close(figure)
assert os.path.isfile(report_path) and os.path.getsize(report_path) > 0
display(Image(filename=report_path))


### Paired literal SQL solution

Rebuild the accepted delivered rows with temporary views and compare the important totals.


In [ ]:
raw.createOrReplaceTempView('run_002_raw')
warehouses.createOrReplaceTempView('s11_sql_warehouses')
carriers.createOrReplaceTempView('s11_sql_carriers')
sql_result = spark.sql("""
WITH numbered AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY shipment_id ORDER BY shipment_id) AS duplicate_position FROM run_002_raw
), typed AS (
  SELECT TRIM(shipment_id) AS shipment_id, run_id,
         TRY_CAST(order_date AS DATE) AS order_date, TRIM(warehouse_id) AS warehouse_id,
         TRIM(carrier_id) AS carrier_id, TRY_CAST(promised_date AS DATE) AS promised_date,
         TRY_CAST(delivered_date AS DATE) AS delivered_date,
         TRY_CAST(shipping_cost AS DECIMAL(12,2)) AS shipping_cost,
         TRY_CAST(package_weight AS DECIMAL(12,2)) AS package_weight,
         UPPER(TRIM(status)) AS status, source_file
  FROM numbered WHERE duplicate_position = 1
), valid_delivered AS (
  SELECT * FROM typed WHERE shipment_id IS NOT NULL AND shipment_id <> ''
    AND order_date IS NOT NULL AND promised_date IS NOT NULL
    AND shipping_cost >= 0 AND package_weight > 0
    AND warehouse_id IS NOT NULL AND warehouse_id <> ''
    AND carrier_id IS NOT NULL AND carrier_id <> ''
    AND status = 'DELIVERED' AND delivered_date IS NOT NULL
)
SELECT v.shipment_id, v.run_id, v.order_date, v.warehouse_id, w.warehouse_name, w.warehouse_city,
       v.carrier_id, c.carrier_name, c.service_type, v.promised_date, v.delivered_date,
       v.shipping_cost, v.package_weight, v.status, v.source_file,
       DATEDIFF(v.delivered_date, v.promised_date) AS delay_days,
       v.delivered_date <= v.promised_date AS on_time,
       DATE_FORMAT(v.delivered_date, 'yyyy-MM') AS delivery_month
FROM valid_delivered v
INNER JOIN s11_sql_warehouses w ON v.warehouse_id = w.warehouse_id
INNER JOIN s11_sql_carriers c ON v.carrier_id = c.carrier_id
ORDER BY v.shipment_id
""")
sql_result.show(truncate=False)
assert sql_result.count() == accepted.count()
sql_cost = sql_result.agg(F.sum('shipping_cost')).first()[0] or Decimal('0')
assert f'{sql_cost:.2f}' == metrics['shipping_cost']
for frame in (raw, quality_rejected, reference_rejected, accepted):
    frame.unpersist()


## Stage 4 — run-003: process TSV, publish and report

Run the complete path from source data to a saved Matplotlib dashboard.


### Step 1 — Read and inspect the source batch

Add the run identifier and source filename before validation.


In [ ]:
run_id = 'run-003'
source_path = f'{PROJECT}/source/run-003/shipments.tsv'
raw = spark.read.schema(shipment_schema).option('header', True).option('sep', '\t').csv(source_path)
raw = raw.withColumn('run_id', F.lit(run_id)).withColumn('source_file', F.input_file_name()).cache()
raw.show(truncate=False)
print('Raw rows:', raw.count())


### Step 2 — Deduplicate, convert and classify

Retain malformed rows with a readable quality reason.


In [ ]:
unique = raw.dropDuplicates(['shipment_id'])
parsed = (unique
    .withColumn('shipment_id', F.trim('shipment_id'))
    .withColumn('warehouse_id', F.trim('warehouse_id'))
    .withColumn('carrier_id', F.trim('carrier_id'))
    .withColumn('status', F.upper(F.trim('status')))
    .withColumn('order_date', F.expr('try_cast(order_date as date)'))
    .withColumn('promised_date', F.expr('try_cast(promised_date as date)'))
    .withColumn('delivered_date', F.expr('try_cast(delivered_date as date)'))
    .withColumn('shipping_cost', F.expr('try_cast(shipping_cost as decimal(12,2))'))
    .withColumn('package_weight', F.expr('try_cast(package_weight as decimal(12,2))')))
quality_reason = F.concat_ws('; ',
    F.when(F.col('shipment_id').isNull() | (F.col('shipment_id') == ''), 'missing shipment id'),
    F.when(F.col('order_date').isNull(), 'invalid order date'),
    F.when(F.col('promised_date').isNull(), 'invalid promised date'),
    F.when(F.col('shipping_cost').isNull() | (F.col('shipping_cost') < 0), 'invalid shipping cost'),
    F.when(F.col('package_weight').isNull() | (F.col('package_weight') <= 0), 'invalid package weight'),
    F.when(F.col('warehouse_id').isNull() | (F.col('warehouse_id') == ''), 'missing warehouse'),
    F.when(F.col('carrier_id').isNull() | (F.col('carrier_id') == ''), 'missing carrier'),
    F.when(F.col('status').isNull() | ~F.col('status').isin('DELIVERED', 'IN_TRANSIT', 'CANCELLED'), 'invalid status'),
    F.when((F.col('status') == 'DELIVERED') & F.col('delivered_date').isNull(), 'missing delivered date'))
classified = parsed.withColumn('reject_reason', quality_reason)
quality_rejected = (classified.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('QUALITY')).cache())
valid = classified.filter(F.col('reject_reason') == '').drop('reject_reason')
delivered = valid.filter(F.col('status') == 'DELIVERED')
print('Unique:', unique.count(), '| quality rejected:', quality_rejected.count(), '| valid:', valid.count(), '| delivered:', delivered.count())


### Step 3 — Join references and calculate delivery measures

Unknown references are rejected; matched deliveries receive delay and on-time values.


In [ ]:
joined = (delivered.join(F.broadcast(warehouses), 'warehouse_id', 'left')
          .join(F.broadcast(carriers), 'carrier_id', 'left'))
reference_reason = F.concat_ws('; ',
    F.when(F.col('warehouse_name').isNull(), 'unknown warehouse'),
    F.when(F.col('carrier_name').isNull(), 'unknown carrier'))
reference_checked = joined.withColumn('reject_reason', reference_reason)
reference_rejected = (reference_checked.filter(F.col('reject_reason') != '').withColumn('reject_stage', F.lit('REFERENCE')).cache())
accepted = (reference_checked.filter(F.col('reject_reason') == '').drop('reject_reason')
    .withColumn('delay_days', F.datediff('delivered_date', 'promised_date'))
    .withColumn('on_time', F.col('delay_days') <= 0)
    .withColumn('delivery_month', F.date_format('delivered_date', 'yyyy-MM'))
    .select('shipment_id', 'run_id', 'order_date', 'warehouse_id', 'warehouse_name', 'warehouse_city',
            'carrier_id', 'carrier_name', 'service_type', 'promised_date', 'delivered_date',
            'shipping_cost', 'package_weight', 'status', 'source_file', 'delay_days', 'on_time', 'delivery_month')
    .cache())
accepted.orderBy("shipment_id").show(truncate=False)
print('Reference rejected:', reference_rejected.count(), '| accepted:', accepted.count())


### Step 4 — Write run outputs and audit metrics

Overwrite only this run so a replay does not duplicate earlier data.


In [ ]:
rejected = quality_rejected.unionByName(reference_rejected, allowMissingColumns=True)
curated_path = f'{PROJECT}/destination/curated_shipments/run_id={run_id}'
rejected_path = f'{PROJECT}/destination/rejected_records/run_id={run_id}'
audit_path = f'{PROJECT}/destination/audit/run_id={run_id}'
accepted.drop('run_id').write.mode('overwrite').partitionBy('delivery_month').parquet(curated_path)
rejected.drop('run_id').write.mode('overwrite').partitionBy('reject_stage').parquet(rejected_path)
accepted_count = accepted.count()
on_time_count = accepted.filter(F.col('on_time')).count()
shipping_cost_total = accepted.agg(F.sum('shipping_cost').alias('value')).first()['value'] or Decimal('0')
metrics = {
    'run_id': run_id, 'raw_rows': raw.count(), 'duplicate_rows': raw.count() - unique.count(),
    'quality_rejected_rows': quality_rejected.count(), 'non_delivered_rows': valid.count() - delivered.count(),
    'reference_rejected_rows': reference_rejected.count(), 'rejected_rows': rejected.count(),
    'accepted_shipments': accepted_count, 'on_time_shipments': on_time_count,
    'late_shipments': accepted_count - on_time_count, 'shipping_cost': f'{shipping_cost_total:.2f}',
}
spark.createDataFrame([metrics]).drop('run_id').write.mode('overwrite').parquet(audit_path)
print(metrics)


### Step 5 — Read the destination and save a Matplotlib dashboard

Spark produces compact summaries; Matplotlib plots only those summary rows.


In [ ]:
destination = spark.read.parquet(curated_path)
status_rows = destination.groupBy('on_time').count().orderBy(F.desc('on_time')).collect()
carrier_rows = (destination.groupBy('carrier_name').agg(F.avg('delay_days').alias('average_delay')).orderBy('carrier_name').collect())
city_rows = (destination.groupBy('warehouse_city').agg(F.sum('shipping_cost').alias('shipping_cost')).orderBy('warehouse_city').collect())
status_labels = ['On time' if row['on_time'] else 'Late' for row in status_rows]
status_values = [row['count'] for row in status_rows]
carrier_labels = [row['carrier_name'] for row in carrier_rows]
carrier_values = [float(row['average_delay']) for row in carrier_rows]
city_labels = [row['warehouse_city'] for row in city_rows]
city_values = [float(row['shipping_cost']) for row in city_rows]
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].axis('off')
on_time_pct = on_time_count / accepted_count * 100 if accepted_count else 0
axes[0, 0].text(0.05, 0.75, f'Accepted shipments\n{accepted_count}', fontsize=16, weight='bold')
axes[0, 0].text(0.05, 0.43, f'On-time rate\n{on_time_pct:.1f}%', fontsize=16, weight='bold')
axes[0, 0].text(0.05, 0.11, f'Shipping cost\n{float(shipping_cost_total):,.2f}', fontsize=16, weight='bold')
axes[0, 1].bar(status_labels, status_values, color=['#2a9d8f', '#e76f51'][:len(status_labels)])
axes[0, 1].set_title('On-time and late shipments')
axes[0, 1].set_ylabel('Shipments')
axes[1, 0].bar(carrier_labels, carrier_values, color='#457b9d')
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('Average delay by carrier')
axes[1, 0].set_ylabel('Days')
axes[1, 0].tick_params(axis='x', rotation=20)
axes[1, 1].bar(city_labels, city_values, color='#f4a261')
axes[1, 1].set_title('Shipping cost by warehouse city')
axes[1, 1].set_ylabel('Cost')
axes[1, 1].tick_params(axis='x', rotation=20)
figure.suptitle(f'Fulfilment dashboard — {run_id}', fontsize=18, weight='bold')
figure.tight_layout(rect=[0, 0, 1, 0.95])
report_path = f'{PROJECT}/reports/{run_id}-dashboard.png'
figure.savefig(report_path, dpi=150, bbox_inches='tight')
plt.close(figure)
assert os.path.isfile(report_path) and os.path.getsize(report_path) > 0
display(Image(filename=report_path))


### Paired literal SQL solution

Rebuild the accepted delivered rows with temporary views and compare the important totals.


In [ ]:
raw.createOrReplaceTempView('run_003_raw')
warehouses.createOrReplaceTempView('s11_sql_warehouses')
carriers.createOrReplaceTempView('s11_sql_carriers')
sql_result = spark.sql("""
WITH numbered AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY shipment_id ORDER BY shipment_id) AS duplicate_position FROM run_003_raw
), typed AS (
  SELECT TRIM(shipment_id) AS shipment_id, run_id,
         TRY_CAST(order_date AS DATE) AS order_date, TRIM(warehouse_id) AS warehouse_id,
         TRIM(carrier_id) AS carrier_id, TRY_CAST(promised_date AS DATE) AS promised_date,
         TRY_CAST(delivered_date AS DATE) AS delivered_date,
         TRY_CAST(shipping_cost AS DECIMAL(12,2)) AS shipping_cost,
         TRY_CAST(package_weight AS DECIMAL(12,2)) AS package_weight,
         UPPER(TRIM(status)) AS status, source_file
  FROM numbered WHERE duplicate_position = 1
), valid_delivered AS (
  SELECT * FROM typed WHERE shipment_id IS NOT NULL AND shipment_id <> ''
    AND order_date IS NOT NULL AND promised_date IS NOT NULL
    AND shipping_cost >= 0 AND package_weight > 0
    AND warehouse_id IS NOT NULL AND warehouse_id <> ''
    AND carrier_id IS NOT NULL AND carrier_id <> ''
    AND status = 'DELIVERED' AND delivered_date IS NOT NULL
)
SELECT v.shipment_id, v.run_id, v.order_date, v.warehouse_id, w.warehouse_name, w.warehouse_city,
       v.carrier_id, c.carrier_name, c.service_type, v.promised_date, v.delivered_date,
       v.shipping_cost, v.package_weight, v.status, v.source_file,
       DATEDIFF(v.delivered_date, v.promised_date) AS delay_days,
       v.delivered_date <= v.promised_date AS on_time,
       DATE_FORMAT(v.delivered_date, 'yyyy-MM') AS delivery_month
FROM valid_delivered v
INNER JOIN s11_sql_warehouses w ON v.warehouse_id = w.warehouse_id
INNER JOIN s11_sql_carriers c ON v.carrier_id = c.carrier_id
ORDER BY v.shipment_id
""")
sql_result.show(truncate=False)
assert sql_result.count() == accepted.count()
sql_cost = sql_result.agg(F.sum('shipping_cost')).first()[0] or Decimal('0')
assert f'{sql_cost:.2f}' == metrics['shipping_cost']
for frame in (raw, quality_rejected, reference_rejected, accepted):
    frame.unpersist()


## Stage 5 — Validate all runs and save the combined dashboard

Use only persisted audit and curated Parquet outputs for final validation.


### Step 1 — Compare every audit with the expected metrics

The expected file provides deterministic completion checks.


In [ ]:
with open(f'{PROJECT}/validation/expected_metrics.json', encoding='utf-8') as stream:
    expected = json.load(stream)
for run_id in ['run-001', 'run-002', 'run-003']:
    actual = spark.read.parquet(f'{PROJECT}/destination/audit/run_id={run_id}').first().asDict()
    for key, value in expected[run_id].items():
        assert actual[key] == value, (run_id, key, actual[key], value)
print('All run audits match expected values')


### Step 2 — Read every curated partition together

The `basePath` option recovers `run_id` and `delivery_month` from directory partitions.


In [ ]:
curated_base = f'{PROJECT}/destination/curated_shipments'
all_deliveries = spark.read.option('basePath', curated_base).parquet(curated_base)
all_deliveries.groupBy('run_id', 'delivery_month').count().orderBy('run_id').show()
assert all_deliveries.count() == 15


### Step 3 — Aggregate combined report measures

Collect only the small summaries required by Matplotlib.


In [ ]:
combined_count = all_deliveries.count()
combined_on_time = all_deliveries.filter(F.col('on_time')).count()
combined_cost = all_deliveries.agg(F.sum('shipping_cost')).first()[0] or Decimal('0')
status_rows = all_deliveries.groupBy('on_time').count().orderBy(F.desc('on_time')).collect()
carrier_rows = all_deliveries.groupBy('carrier_name').agg(F.avg('delay_days').alias('average_delay')).orderBy('carrier_name').collect()
city_rows = all_deliveries.groupBy('warehouse_city').agg(F.sum('shipping_cost').alias('shipping_cost')).orderBy('warehouse_city').collect()
assert combined_on_time == 9
assert f'{combined_cost:.2f}' == '2290.00'


### Step 4 — Save and display the combined dashboard

Use the same visual structure as the individual run reports.


In [ ]:
status_labels = ['On time' if row['on_time'] else 'Late' for row in status_rows]
status_values = [row['count'] for row in status_rows]
carrier_labels = [row['carrier_name'] for row in carrier_rows]
carrier_values = [float(row['average_delay']) for row in carrier_rows]
city_labels = [row['warehouse_city'] for row in city_rows]
city_values = [float(row['shipping_cost']) for row in city_rows]
figure, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].axis('off')
axes[0, 0].text(0.05, 0.75, f'Accepted shipments\n{combined_count}', fontsize=16, weight='bold')
axes[0, 0].text(0.05, 0.43, f'On-time rate\n{combined_on_time / combined_count * 100:.1f}%', fontsize=16, weight='bold')
axes[0, 0].text(0.05, 0.11, f'Shipping cost\n{float(combined_cost):,.2f}', fontsize=16, weight='bold')
axes[0, 1].bar(status_labels, status_values, color=['#2a9d8f', '#e76f51'][:len(status_labels)])
axes[0, 1].set_title('On-time and late shipments')
axes[0, 1].set_ylabel('Shipments')
axes[1, 0].bar(carrier_labels, carrier_values, color='#457b9d')
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('Average delay by carrier')
axes[1, 0].set_ylabel('Days')
axes[1, 0].tick_params(axis='x', rotation=20)
axes[1, 1].bar(city_labels, city_values, color='#f4a261')
axes[1, 1].set_title('Shipping cost by warehouse city')
axes[1, 1].set_ylabel('Cost')
axes[1, 1].tick_params(axis='x', rotation=20)
figure.suptitle('Fulfilment dashboard — all runs', fontsize=18, weight='bold')
figure.tight_layout(rect=[0, 0, 1, 0.95])
combined_report_path = f'{PROJECT}/reports/all-runs-dashboard.png'
figure.savefig(combined_report_path, dpi=150, bbox_inches='tight')
plt.close(figure)
assert os.path.isfile(combined_report_path) and os.path.getsize(combined_report_path) > 0
display(Image(filename=combined_report_path))


### Step 5 — Verify all four report files and inspect the final plan

The project is complete only when every saved report exists.


In [ ]:
report_files = [f'{PROJECT}/reports/run-001-dashboard.png', f'{PROJECT}/reports/run-002-dashboard.png',
                f'{PROJECT}/reports/run-003-dashboard.png', f'{PROJECT}/reports/all-runs-dashboard.png']
for report_file in report_files:
    assert os.path.isfile(report_file) and os.path.getsize(report_file) > 0, report_file
print('Saved reports:', *report_files, sep='\n- ')
all_deliveries.groupBy('carrier_name').agg(F.avg('delay_days')).explain('formatted')


### Paired literal SQL solution

Summarize the complete curated destination with SQL and compare final totals.


In [ ]:
all_deliveries.createOrReplaceTempView('s11_all_deliveries')
sql_result = spark.sql("""
SELECT carrier_name, COUNT(*) AS shipments,
       SUM(CASE WHEN on_time THEN 1 ELSE 0 END) AS on_time_shipments,
       SUM(shipping_cost) AS shipping_cost, AVG(delay_days) AS average_delay
FROM s11_all_deliveries GROUP BY carrier_name ORDER BY carrier_name
""")
sql_result.show()
assert sql_result.agg(F.sum('shipments')).first()[0] == 15
assert f"{sql_result.agg(F.sum('shipping_cost')).first()[0]:.2f}" == '2290.00'
